# 08b · Integración con la BD común: `BD_modelo_fly.xlsx` y `recomendar(user_id)`

Etapa 8, paso b (local). Requiere el paso 8a (pesos del DeepFM en `models/deepfm_pesos.npz`).

1. **Arma el libro Excel** que hace de base de datos común (revisión §12.8): prendas, outfits, usuarios, clósets, historial, candidatos, predicciones, métricas, impacto ODS y diccionario de datos.
2. **Comprueba** que lo que se lee del Excel da los mismos números que los CSV del proyecto y que el DeepFM en numpy reproduce al de PyTorch.
3. **`recomendar(user_id)`** (`src/modelo_fly/recomendador.py`): lee del Excel el clóset y el historial del usuario, arma outfits nuevos, los pasa por las dos fases (DeepFM → Random Forest, el mejor sistema del paso 7d) y **escribe** en las hojas `candidatos` y `predicciones`. Se muestra el antes y el después como evidencia de integración.
4. **Impacto ODS 12** con indicadores medibles.

In [ ]:
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter
from sklearn.metrics import mean_absolute_error, roc_auc_score

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")
sys.path.insert(0, str(ROOT / "src"))

from modelo_fly.caracteristicas import codificar, codificar_usuarios
from modelo_fly.ranking import dos_etapas, matrices
from modelo_fly.recomendador import ALFA, BETA, EXPERIMENTO, TOP, cargar_modelos, guardar, leer_bd, recomendar
from modelo_fly.usuarios import ESCALA, cargar_items, compatibilidad_polyvore, rasgos, utilidad

PROC = ROOT / "data" / "processed" / "hito3"
RES = ROOT / "results" / "hito3"
RUTA = ROOT / "data" / "BD_modelo_fly.xlsx"
AHORA = datetime.now().isoformat(timespec="seconds")
t0 = time.time()

## 1. Modelos y comprobaciones

In [ ]:
deepfm, rf_ttmn, rf_dlmn = cargar_modelos(ROOT)
items, atrib, outfits = cargar_items(PROC)
usuarios = pd.read_csv(PROC / "usuarios.csv")
historial = pd.read_csv(PROC / "historial.csv")
cand = pd.read_csv(PROC / "candidatos.csv").merge(
    pd.read_csv(PROC / "deepfm_pred_candidatos.csv.gz")[["user_id", "outfit_id", "deepfm_s0"]], on=["user_id", "outfit_id"])
fila_u, _ = codificar_usuarios(usuarios)

# a) DeepFM en numpy vs PyTorch (mismas predicciones del paso 8a sobre los candidatos de test)
ver = pd.read_csv(ROOT / "models" / "deepfm_verificacion.csv.gz").merge(cand, on=["user_id", "outfit_id"])
p_np = deepfm.predecir(np.array([fila_u[u] for u in ver["user_id"]]), codificar(ver["item_ids"].str.split("|").tolist(), atrib))
print(f"DeepFM numpy vs PyTorch: diferencia máxima {np.abs(p_np - ver['p_torch']).max():.2e} en {len(ver)} candidatos")
assert np.allclose(p_np, ver["p_torch"], atol=1e-4)

# b) Random Forest reentrenado con los hiperparámetros del paso 7b (en test de la fase 2)
te = outfits["particion"] == "test"
X_te = codificar(outfits.loc[te, "ids"].tolist(), atrib)
print(f"Random Forest: AUC TTMn {roc_auc_score(outfits.loc[te, 'real'], rf_ttmn.predict_proba(X_te)[:, 1]):.3f} "
      f"(paso 7b: 0,623) | MAE DLMn {mean_absolute_error(outfits.loc[te, 'score'], rf_dlmn.predict(X_te)):.3f} (paso 7b: 0,190)")

## 2. Tablas del libro

Las hojas `candidatos` y `predicciones` empiezan con la evaluación del paso 7 (el ranking del experimento 3 sobre los candidatos de valid y test, semilla 0); `recomendar()` les agrega filas con `origen = recomendar`.

In [ ]:
X_c = codificar(cand["item_ids"].str.split("|").tolist(), atrib)
cand["p_ttmn"] = rf_ttmn.predict_proba(X_c)[:, 1]
cand["dlmn"] = rf_dlmn.predict(X_c)
partes = []
for particion, d in cand.groupby("particion"):
    d = d.sort_values(["user_id", "outfit_id"]).reset_index(drop=True)
    u, Y, P = matrices(d, "deepfm_s0")
    final = dos_etapas(P, matrices(d, "p_ttmn")[2], matrices(d, "dlmn")[2], ALFA, BETA, top=int(TOP * P.shape[1]))
    d["score_final"] = final.ravel()
    partes.append(d)
cand = pd.concat(partes, ignore_index=True)
cand["en_top"] = (cand["score_final"] >= 2).astype(int)
cand["rango"] = cand.groupby(["particion", "user_id"])["score_final"].rank(ascending=False, method="first").astype(int)
cand["score_final"] = np.where(cand["en_top"] == 1, cand["score_final"] - 2, np.nan)

TABLAS = {
    "items": items,
    "outfits": outfits.assign(item_ids=outfits["ids"].str.join("|"))[
        ["outfit_id", "tipo", "real", "set_id_origen", "particion", "n_items", "cambiadas", "item_ids",
         *[c for c in outfits.columns if c.startswith(("aplica_", "cumple_"))], "score"]].rename(columns={"score": "score_reglas"}),
    "usuarios": usuarios,
    "closets": pd.read_csv(PROC / "closets.csv"),
    "historial": historial[["user_id", "orden", "fecha", "outfit_id", "item_ids", "n_items", "particion",
                            *[c for c in historial.columns if c.startswith("viola_")], "score_reglas", "p_real", "aceptado"]],
    "candidatos": cand.rename(columns={"deepfm_s0": "p_deepfm"}).assign(fecha=AHORA)[
        ["user_id", "outfit_id", "item_ids", "particion", "origen", "n_items", "p_real", "aceptado", "p_deepfm", "en_top", "fecha"]],
    "predicciones": cand[cand["rango"] <= 10].rename(columns={"deepfm_s0": "p_deepfm"}).assign(
        experimento=EXPERIMENTO, origen="evaluacion_" + cand["particion"], fecha=AHORA)[
        ["user_id", "outfit_id", "item_ids", "experimento", "p_deepfm", "p_ttmn", "dlmn", "score_final", "rango",
         "aceptado", "origen", "fecha"]].sort_values(["origen", "user_id", "rango"]),
}
for c in ["p_deepfm", "p_ttmn", "score_final"]:
    TABLAS["predicciones"][c] = TABLAS["predicciones"][c].round(5)
TABLAS["predicciones"]["dlmn"] = TABLAS["predicciones"]["dlmn"].round(3)
TABLAS["candidatos"]["p_deepfm"] = TABLAS["candidatos"]["p_deepfm"].round(5)

# Métricas en formato largo: fase 2 (paso 7b) y ranking final (paso 7d)
f2 = pd.read_csv(RES / "fase2" / "fase2_metricas.csv")
f2 = f2.melt(id_vars=["modelo", "experimento", "semilla"], value_vars=["AUC_TTMn", "AUC_TTMn_aleatorio", "AUC_TTMn_dificil", "MAE_DLMn"],
             var_name="metrica", value_name="valor").assign(fuente="07b · fase 2 (outfits de Polyvore, test)")
NOMBRES = {"1": "DeepFM solo", "2": "DeepFM + XGBoost", "3": "DeepFM + Random Forest", "4": "DeepFM + GFS (A)",
           "5": "DeepFM + MLP", "6": "DeepFM + GFS barajada", "anexo": "DeepFM + GFS (B)", "azar": "azar", "techo": "techo (p_real)"}
rk = pd.read_csv(RES / "ranking" / "ranking_metricas_por_semilla.csv", dtype={"experimento": str})
rk = rk.melt(id_vars=["experimento", "semilla"], var_name="metrica", value_name="valor").assign(
    modelo=lambda d: d["experimento"].map(NOMBRES), fuente="07d · ranking final (candidatos de test, 350 usuarios)")
TABLAS["metricas"] = pd.concat([f2, rk], ignore_index=True)[["fuente", "experimento", "modelo", "semilla", "metrica", "valor"]]
TABLAS["metricas"]["valor"] = TABLAS["metricas"]["valor"].round(5)
TABLAS["metricas"]["experimento"] = TABLAS["metricas"]["experimento"].astype(str)
display(pd.Series({h: len(t) for h, t in TABLAS.items()}, name="filas").to_frame().T)

## 3. Impacto ODS 12 (producción y consumo responsables)

El sistema solo arma outfits con prendas que el usuario **ya tiene**: la recomendación compite con la compra de ropa nueva. Se mide con `recomendar()` sobre los 350 usuarios (3 recomendaciones cada uno, sin escribir en el libro) y con el gusto simulado como juez, porque es la única "opinión" disponible de estos usuarios. Comparación: 3 outfits al azar del mismo clóset (lo que haría alguien sin recomendador) y el top 3 del DeepFM solo.

In [ ]:
compat, c_media, c_desv = compatibilidad_polyvore(outfits, atrib)
perfil = usuarios.set_index("user_id")
bd_mem = {"usuarios": usuarios.astype({"user_id": str}), "closets": TABLAS["closets"],
          "items": items.fillna(""), "historial": historial}

def p_aceptar(u, ids):
    usr = perfil.loc[u].to_dict() | {"user_id": u}
    r = rasgos([atrib[i] for i in ids], usr["clima"], compat)
    return 1 / (1 + np.exp(-ESCALA * (utilidad(r, usr, c_media, c_desv)[0] - usr["sesgo"])))

usadas = historial[historial["aceptado"] == 1].groupby("user_id")["item_ids"].agg(lambda s: set("|".join(s).split("|")))
filas = []
t = time.time()
for u in usuarios["user_id"]:
    c, r = recomendar(u, None, deepfm, rf_ttmn, rf_dlmn, semilla=0, escribir=False, bd=bd_mem)
    ids_rec = [x.split("|") for x in r["item_ids"]]
    azar = c.sample(3, random_state=0)["item_ids"].str.split("|").tolist()
    solo_dfm = c.nlargest(3, "p_deepfm")["item_ids"].str.split("|").tolist()
    prendas = set().union(*map(set, ids_rec))
    filas.append({"user_id": u, "grupo": perfil.loc[u, "grupo"],
                  "p_rec": np.mean([p_aceptar(u, i) for i in ids_rec]),
                  "p_azar": np.mean([p_aceptar(u, i) for i in azar]),
                  "p_deepfm": np.mean([p_aceptar(u, i) for i in solo_dfm]),
                  "prendas_rec": len(prendas), "repetidas": sum(map(len, ids_rec)) - len(prendas),
                  "olvidadas": len(prendas - usadas.get(u, set())) / len(prendas),
                  "closet": perfil.loc[u, "n_closet"]})
imp = pd.DataFrame(filas)
print(f"{len(imp)} usuarios en {time.time() - t:.0f} s")
display(imp.groupby("grupo")[["p_rec", "p_deepfm", "p_azar", "olvidadas"]].mean().round(3))

In [ ]:
def pct(x):
    return f"{100 * x:.1f} %"

TABLAS["impacto_ods"] = pd.DataFrame([
    {"indicador": "Outfits recomendados hechos solo con prendas del clóset", "valor": "100 %",
     "meta_ods": "12.5 (reducir residuos: reutilizar)", "como_se_mide": "Por diseño: recomendar() solo combina la hoja closets del usuario"},
    {"indicador": "Probabilidad media de que el usuario acepte una recomendación", "valor": pct(imp["p_rec"].mean()),
     "meta_ods": "12.2 (uso eficiente de recursos)", "como_se_mide": "Gusto simulado sobre las 3 recomendaciones de cada usuario (350 usuarios)"},
    {"indicador": "… si elige 3 outfits al azar de su clóset (sin recomendador)", "valor": pct(imp["p_azar"].mean()),
     "meta_ods": "Línea base", "como_se_mide": "Misma medida sobre 3 outfits al azar de sus 200 candidatos"},
    {"indicador": "… con el DeepFM solo (sin fase 2)", "valor": pct(imp["p_deepfm"].mean()),
     "meta_ods": "Comparación", "como_se_mide": "Misma medida sobre el top 3 del DeepFM"},
    {"indicador": "Mejora relativa de la aceptación frente al azar", "valor": f"+{100 * (imp['p_rec'].mean() / imp['p_azar'].mean() - 1):.0f} %",
     "meta_ods": "12.2", "como_se_mide": "p_rec / p_azar − 1"},
    {"indicador": "Prendas distintas puestas en uso por usuario (3 outfits)", "valor": f"{imp['prendas_rec'].mean():.1f} de {imp['closet'].mean():.0f}",
     "meta_ods": "12.5", "como_se_mide": "Los 3 outfits no repiten prendas (repetidas = " + str(int(imp["repetidas"].sum())) + ")"},
    {"indicador": "Prendas recomendadas que el usuario nunca había usado en un outfit aceptado", "valor": pct(imp["olvidadas"].mean()),
     "meta_ods": "12.5 (alargar la vida útil de las prendas)", "como_se_mide": "Prendas de las recomendaciones ausentes de su historial aceptado"},
    {"indicador": "Usuarios con clóset 100 % catálogo propio: aceptación", "valor": pct(imp.loc[imp["grupo"] == "transferencia", "p_rec"].mean()),
     "meta_ods": "12.8 (información para consumo responsable)", "como_se_mide": "Igual que arriba, solo los 50 usuarios de transferencia"},
])
imp.round(4).to_csv(RES / "bd_comun_impacto_por_usuario.csv", index=False)
display(TABLAS["impacto_ods"])

## 4. Diccionario de datos y escritura del libro

In [ ]:
DESCRIPCION = {
    "item_id": "ID de la prenda: 'pv_' + ID de Polyvore, o 'P' + número del catálogo propio",
    "origen": "items: polyvore / propio · closets: de dónde viene la prenda · candidatos y predicciones: historial / nuevo (evaluación) o recomendar (escrito por recomendar())",
    "particion": "train / valid / test (en historial y candidatos es temporal por usuario; en outfits e items es por outfit de Polyvore)",
    "name": "Texto original de la prenda (nombre en Polyvore o descripción del catálogo propio)",
    "slot": "Lugar en el outfit: arriba, abajo, vestido, exterior, calzado, bolso, accesorio, joyeria",
    "tono": "Tono del color (claro, medio, oscuro…)", "familia_color": "Familia de color (neutro, cálido, frío…)",
    "patron": "Patrón (liso, rayas, cuadros, floral…)", "material": "Material principal",
    "estilo": "Estilo (clasico, minimalista, vintage, atrevido, deportivo…)", "calce": "Calce (ajustado, regular, oversize)",
    "epoca": "Década o época que evoca", "largo": "Largo de la prenda (corto, medio, largo)",
    "outfit_id": "ID del outfit (Polyvore: set_id y sufijo del negativo; usuarios: U###_O## historial, U###_V/T### candidatos, U###_R<fecha>_### recomendar())",
    "tipo": "real (outfit armado por una persona en Polyvore) o el tipo de negativo (aleatorio, dificil)",
    "real": "1 = outfit real de Polyvore, 0 = negativo. Etiqueta de TTMn (fase 2)",
    "set_id_origen": "Outfit real del que sale el negativo (agrupa los folds de la validación cruzada)",
    "n_items": "Cantidad de prendas del outfit", "cambiadas": "Prendas cambiadas para armar el negativo",
    "item_ids": "Prendas del outfit separadas por '|' (llaves de la hoja items)",
    "score_reglas": "Reglas de las 7 que cumple el outfit (0–7). Etiqueta de DLMn (fase 2)",
    "user_id": "ID del usuario simulado (U001–U350)",
    "grupo": "mixto (clóset 70 % Polyvore / 30 % propio) o transferencia (clóset 100 % catálogo propio)",
    "arquetipo": "OCULTO al modelo. Gusto programado en la simulación (minimalista, clasico, vintage, atrevido, deportivo)",
    "estilo_fav": "Estilo favorito declarado al registrarse", "edad": "Edad en años (el modelo la usa por tramos 18-24, 25-34, 35-45)",
    "genero": "Género declarado", "ciudad": "Ciudad del Perú", "clima": "Clima de la ciudad (frio, templado, calido)",
    "n_closet": "Prendas en el clóset del usuario",
    "b_estilo": "OCULTO. Peso del estilo favorito en el gusto simulado", "d_neutros": "OCULTO. Peso de los colores neutros",
    "c_clima": "OCULTO. Peso de vestir acorde al clima", "e_compat": "OCULTO. Peso de la compatibilidad humana (aprendida de Polyvore)",
    "ruido": "OCULTO. Fracción de decisiones invertidas al azar (5–10 %)", "sesgo": "OCULTO. Mediana de la utilidad del usuario (centra su tasa de aceptación)",
    "orden": "Posición de la interacción en el tiempo (0–49)", "fecha": "Fecha de la interacción o de la escritura",
    "p_real": "Probabilidad de aceptar según la simulación (verdad oculta; solo para evaluar y como techo)",
    "aceptado": "1 = el usuario aceptó el outfit, 0 = lo rechazó. Etiqueta del DeepFM y de las métricas de ranking",
    "p_deepfm": "Fase 1: probabilidad de que el usuario lo acepte según el DeepFM",
    "en_top": "1 = el outfit pasó el filtro del DeepFM (top 25 % del usuario)",
    "experimento": "Sistema que generó la predicción (3 · DeepFM + Random Forest, el mejor del paso 7d)",
    "p_ttmn": "Fase 2: probabilidad de que el outfit sea real/compatible (salida TTMn, Random Forest)",
    "dlmn": "Fase 2: armonía predicha 0–7 (salida DLMn, Random Forest)",
    "score_final": "α·pct(DeepFM) + (1 − α)·[β·pct(TTMn) + (1 − β)·pct(DLMn)], con α = 0,6 y β = 0,5 (elegidos en valid)",
    "rango": "Posición en el ranking final del usuario (1 = primera recomendación)",
    "fuente": "Notebook o paso que produjo la métrica", "modelo": "Modelo o sistema evaluado", "semilla": "Semilla de entrenamiento",
    "metrica": "Nombre de la métrica", "valor": "Valor de la métrica",
    "indicador": "Indicador de impacto", "meta_ods": "Meta del ODS 12 a la que aporta", "como_se_mide": "Cálculo",
    "hoja": "Hoja del libro", "columna": "Columna", "tipo_dato": "Tipo de dato", "descripcion": "Significado",
    "contenido": "Qué guarda la hoja", "filas": "Filas al construir el libro", "quien_escribe": "Proceso que escribe la hoja",
}
PATRONES = {"w_": "OCULTO. Peso de la regla {} en el gusto simulado (cuánto le molesta romperla)",
            "viola_": "1 si el outfit rompe la regla {}", "aplica_": "1 si la regla {} aplica al outfit",
            "cumple_": "1 si el outfit cumple la regla {}"}

def describir(col):
    for pre, texto in PATRONES.items():
        if col.startswith(pre):
            return texto.format(col[len(pre):])
    return DESCRIPCION.get(col, "")

LEEME = {
    "items": ("Prendas con sus atributos (Polyvore + catálogo propio)", "Preparación (etapas 2–4)"),
    "outfits": ("Outfits de Polyvore (reales y negativos) con las 7 reglas: datos de la fase 2", "Preparación (etapa 4)"),
    "usuarios": ("Usuarios simulados: datos de registro y parámetros ocultos del gusto", "Simulación (etapa 6)"),
    "closets": ("Prendas de cada usuario", "Simulación (etapa 6)"),
    "historial": ("50 outfits por usuario con su decisión (35 train, 7 valid, 8 test)", "Simulación (etapa 6)"),
    "candidatos": ("Outfits que se ordenan para cada usuario, con la probabilidad del DeepFM", "Etapa 7 y recomendar()"),
    "predicciones": ("Ranking final (top 10 de evaluación y 3 recomendaciones por llamada)", "Etapa 7 y recomendar()"),
    "metricas": ("Métricas de la fase 2 y del ranking final por semilla", "Etapa 7"),
    "impacto_ods": ("Indicadores de impacto ODS 12", "Etapa 8"),
    "diccionario": ("Significado de cada columna", "Etapa 8"),
}
dic = pd.DataFrame([{"hoja": h, "columna": c, "tipo_dato": str(t[c].dtype).replace("object", "texto"), "descripcion": describir(c)}
                    for h, t in TABLAS.items() for c in t.columns])
dic = pd.concat([dic, pd.DataFrame([{"hoja": "diccionario", "columna": c, "tipo_dato": "texto", "descripcion": describir(c)}
                                    for c in dic.columns])], ignore_index=True)
assert (dic["descripcion"] != "").all(), dic.loc[dic["descripcion"] == "", ["hoja", "columna"]]
TABLAS["diccionario"] = dic
leeme = pd.DataFrame([{"hoja": h, "contenido": c, "filas": len(TABLAS[h]), "quien_escribe": q} for h, (c, q) in LEEME.items()])

t = time.time()
with pd.ExcelWriter(RUTA, engine="openpyxl") as w:
    leeme.to_excel(w, sheet_name="leeme", index=False)
    for h, tabla in TABLAS.items():
        tabla.to_excel(w, sheet_name=h, index=False)
    for ws in w.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for celda in ws[1]:
            celda.font = Font(bold=True, color="FFFFFF")
            celda.fill = PatternFill("solid", fgColor="1F6FB4")
        for i, col in enumerate(ws.iter_cols(min_row=1, max_row=min(ws.max_row, 200)), start=1):
            ws.column_dimensions[get_column_letter(i)].width = min(60, max(10, max(len(str(c.value or "")) for c in col) + 2))
print(f"Libro escrito en {time.time() - t:.0f} s: {RUTA.relative_to(ROOT)} ({RUTA.stat().st_size / 1e6:.1f} MB)")
display(leeme)

lineas = ["# Diccionario de datos · `data/BD_modelo_fly.xlsx`", "",
          "Generado por `notebooks/hito3/08b_bd_comun.ipynb`. Las columnas marcadas OCULTO son parámetros de la simulación: "
          "el modelo no las ve; sirven para evaluar.", ""]
for h, (contenido, quien) in LEEME.items():
    lineas += [f"## `{h}`", "", f"{contenido}. Escribe: {quien}. Filas: {len(TABLAS[h]):,}.".replace(",", " "), "",
               "| Columna | Tipo | Descripción |", "|---|---|---|"]
    lineas += [f"| `{r.columna}` | {r.tipo_dato} | {r.descripcion} |" for r in dic[dic["hoja"] == h].itertuples()]
    lineas.append("")
(ROOT / "docs" / "DICCIONARIO_DATOS.md").write_text("\n".join(lineas), encoding="utf-8")

## 5. Evidencia de integración: `recomendar(user_id)` lee y escribe el Excel

Tres usuarios de perfiles distintos. `recomendar()` lee el libro y devuelve sus filas; `guardar()` las agrega a las hojas `candidatos` y `predicciones` (una sola escritura para los tres: abrir y guardar un libro de 15 MB tarda). Antes y después se cuentan las filas.

In [ ]:
def filas_hojas():
    libro = load_workbook(RUTA, read_only=True)
    n = {h: libro[h].max_row - 1 for h in ["candidatos", "predicciones"]}
    libro.close()
    return n

antes = filas_hojas()
bd = leer_bd(RUTA)
nombre = items.set_index("item_id")["name"]
DEMO = ["U001", usuarios.loc[usuarios["arquetipo"] == "minimalista", "user_id"].iloc[0],
        usuarios.loc[usuarios["grupo"] == "transferencia", "user_id"].iloc[0]]
lotes = []
for i, u in enumerate(DEMO):
    c, r = recomendar(u, RUTA, deepfm, rf_ttmn, rf_dlmn, semilla=i, escribir=False, bd=bd)
    lotes.append((c, r))
    p = perfil.loc[u]
    print(f"\n{u} · {p['grupo']} · estilo favorito {p['estilo_fav']} · {p['ciudad']} ({p['clima']})")
    for x in r.itertuples():
        print(f"  {x.rango}. score {x.score_final:.3f} | DeepFM {x.p_deepfm:.2f} | TTMn {x.p_ttmn:.2f} | DLMn {x.dlmn:.1f}")
        print("     " + " + ".join(str(nombre.get(i, i))[:28] for i in x.item_ids.split("|")))
guardar(RUTA, pd.concat([c for c, _ in lotes]), pd.concat([r for _, r in lotes]))
despues = filas_hojas()
display(pd.DataFrame({"antes": antes, "después": despues}).assign(agregadas=lambda d: d["después"] - d["antes"]))
nuevas = pd.read_excel(RUTA, sheet_name="predicciones").query("origen == 'recomendar'")
display(nuevas.tail(9))
print(f"Total: {(time.time() - t0) / 60:.1f} min")